In [ ]:
# Imports and Constants
import pandas as pd
import numpy as np
import sqlite3
import joblib
import torch
from torch import Tensor

from climb_conversion import ClimbsFeatureScaler

DB_PATH = "data/storage.db"
SCALER_WEIGHTS_PATH = 'data/weights/scaler-weights.joblib'
DDPM_WEIGHTS_PATH = 'data/weights/ddpm-weights.pth'
LAYOUT_ID = 'layout-0aa86d03949f'

FEATURE_WEIGHTS = [1.0,1.0,1.0,1.0,0.5,0.5,0.5]

GRADE_TO_DIFF = {
    "font": {
        "4a": 10, "4b": 11, "4c": 12,
        "5a": 13, "5b": 14, "5c": 15,
        "6a": 16, "6a+": 17, "6b": 18, "6b+": 19,
        "6c": 20, "6c+": 21,
        "7a": 22, "7a+": 23, "7b": 24, "7b+": 25,
        "7c": 26, "7c+": 27,
        "8a": 28, "8a+": 29, "8b": 30, "8b+": 31,
        "8c": 32, "8c+": 33,
    },
    "v_grade": {
        "V0-": 10, "V0": 11, "V0+": 12,
        "V1": 13, "V1+": 14, "V2": 15,
        "V3": 16, "V3+": 17, "V4": 18, "V4+": 19,
        "V5": 20, "V5+": 21, "V6": 22, "V6+": 22.5,
        "V7": 23, "V7+": 23.5, "V8": 24, "V8+": 25,
        "V9": 26, "V9+": 26.5, "V10": 27, "V10+": 27.5,
        "V11": 28, "V11+": 28.5, "V12": 29, "V12+": 29.5,
        "V13": 30, "V13+": 30.5, "V14": 31, "V14+": 31.5,
        "V15": 32, "V15+": 32.5, "V16": 33,
    },
}

In [ ]:
# ---------------------------------------------------------------------------
# ClimbDDPMGenerator
# ---------------------------------------------------------------------------

class ClimbDDPMGenerator():
    def __init__(
            self,
            scaler: ClimbsFeatureScaler,
            ddpm: ClimbDDPM,
            feature_weights =  torch.tensor(FEATURE_WEIGHTS)
        ):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.scaler = scaler
        self.ddpm = ddpm
        self._cond_cache = {}
        self.holds_manifolds = {}
        self.holds_lookup = {}
        self.deterministic_noise_generator = torch.Generator(device=self.device)
        self.feature_weights = feature_weights
        print(self.feature_weights.shape)

        try:
            with sqlite3.connect(DB_PATH) as conn:
                holds = pd.read_sql_query("SELECT hold_index, x, y, pull_x, pull_y, useability, is_foot, layout_id FROM holds", conn)
                layout_ids = list(set(holds['layout_id'].values))
            
            scaled_holds = self.scaler.transform_hold_features(holds, to_df=True)
            
            for layout_id in layout_ids:
                df = scaled_holds[scaled_holds['layout_id']==layout_id]
                self.holds_manifolds[layout_id] = torch.tensor(df[['x','y','pull_x','pull_y']].values, dtype=torch.float32)
                self.holds_lookup[layout_id] = df['hold_index'].values
                self.holds_lookup[layout_id] = self.holds_lookup[layout_id]
        except Exception as e:
            pass
    
    def log_hold_means(self, layout_id: str | None = None):
        """Log the hold means for each wall."""
        for k, manifold in self.holds_manifolds.items():
            if layout_id == None or layout_id == k:
                means = torch.mean(manifold, dim=0)
                print(f"Wall-id--{k}; Means-- x:{means[0].item()}, y:{means[1].item()}, Px:{means[2].item()}, Py:{means[3].item()} ")

    def _build_cond_tensor(self, n, grade, diff_scale, angle):
        cache_key = (grade, diff_scale, angle)
        if cache_key not in self._cond_cache:
            diff = GRADE_TO_DIFF[diff_scale][grade]
            row = np.array([[diff, 3.0, 1000, float(angle)]])
            row[:, 1] = np.log(1 - (row[:, 1] - 3))     # quality
            row[:, 2] = np.log(row[:, 2])               # ascents
            scaled = self.scaler.cond_features_scaler.transform(row)
            self._cond_cache[cache_key] = scaled[0]
        base = self._cond_cache[cache_key]
        tiled = np.tile(base, (n,1))
        return torch.tensor(tiled, device=self.device, dtype=torch.float32)
    
    def _project_onto_manifold(self, gen_climbs: Tensor, offset_manifold: Tensor)-> Tensor:
        """
            Project each generated hold to its nearest neighbor on the hold manifold.
            
            Args:
                gen_climbs: (B, S, H) predicted clean holds
                return_indices: (boolean) Whether to return the hold indices or hold feature coordinates
            Returns:
                projected: (B, S, H) each hold snapped to nearest manifold point
        """
        B, S, H = gen_climbs.shape
        flat_climbs = gen_climbs.reshape(-1,H)
        dists = torch.cdist(flat_climbs[:,:,:(H-NUM_ROLES)]*self.feature_weights, offset_manifold*self.feature_weights)
        idx = dists.argmin(dim=1)
        return offset_manifold[idx].reshape(B, S, -1)
    
    def _get_offset_manifold(self, layout_id: str, x_offset: float | None)-> Tensor:
        """Method for offsetting the current holds-manifold such that mean-x and mean-y is 0"""
        offset_manifold = self.holds_manifolds[layout_id].clone()
        means = torch.mean(offset_manifold, dim=0).round(decimals=3)
        if x_offset is None:
            x_offset = -(means[0].item())
        y_offset = -(means[1].item())
        
        offset_manifold[:,0] += x_offset
        offset_manifold[:,1] += y_offset
        means = torch.mean(offset_manifold, dim=0)

        return offset_manifold

    def _find_optimal_translation(
        self,
        gen_climbs: Tensor,           # (B, S, H)
        offset_manifold: Tensor,      # (M, H - NUM_ROLES)
        x_offsets: Tensor,            # (Nx,)
        y_offsets: Tensor,            # (Ny,)
    ) -> tuple[Tensor, Tensor]:
        """
        Find the (dx, dy) translation per batch item that minimises total
        nearest-neighbour projection distance onto the hold manifold.

        Returns:
            best_dx: (B,)  optimal x translation for each climb
            best_dy: (B,)  optimal y translation for each climb
        """
        B, S, H = gen_climbs.shape
        
        # Create a null mask so that the null-hold positions don't contribute to the distance metric.
        nonnull_mask = (gen_climbs[:,:,-1] < 0.95).float()
        Nx = x_offsets.shape[0]
        Ny = y_offsets.shape[0]

        # Build a (Nx*Ny, H) manifold shift table — only x/y cols (0 and 2) move
        # Shape: (Nx, Ny, H)
        shifts = torch.zeros(Nx, Ny, H, device=gen_climbs.device)
        shifts[:, :, 0] = x_offsets.unsqueeze(1)   # broadcast over Ny
        shifts[:, :, 1] = y_offsets.unsqueeze(0)   # broadcast over Nx
        G = Nx * Ny
        shifts = shifts.reshape(G, H)         # (G, H)  G = grid size

        # Translate climbs: (B, S, H) + (G, H) -> (G, B, S, H)
        translated = gen_climbs.unsqueeze(0) + shifts.unsqueeze(1).unsqueeze(2)

        # Flatten holds dim for cdist: (G*B, S, H)
        flat = translated.reshape(G * B, S, H)

        # Nearest-neighbour distances to manifold: (G*B, S, M) -> (G*B, S)
        dists = torch.cdist(flat[:,:,:(H-NUM_ROLES)]*self.feature_weights, offset_manifold*self.feature_weights)              # (G*B, S, M)
        nn_dists = dists.min(dim=2).values                      # (G*B, S)
        batch_dist = nn_dists.reshape(G, B, S)                  # (G, B, S)
        batch_dist = nonnull_mask.unsqueeze(0) * batch_dist

        total_dist = batch_dist.sum(dim=2)                      # (G, B)

        # Best grid point per batch item
        best_g = total_dist.argmin(dim=0)                       # (B,)
        best_dx = x_offsets[best_g // Ny]
        best_dy = y_offsets[best_g % Ny]

        return best_dx, best_dy

    def _project_onto_indices_with_translation(
        self,
        gen_climbs: Tensor,
        offset_manifold: Tensor,
        layout_id: str,
        x_offsets: Tensor | None = None,
        y_offsets: Tensor | None = None,
    ) -> list[list[list[int]]]:

        if x_offsets is None:
            x_offsets = torch.linspace(-0.5, 0.5, 51, device=gen_climbs.device)
        if y_offsets is None:
            y_offsets = torch.linspace(-0.5, 0.5, 51, device=gen_climbs.device)

        best_dx, best_dy = self._find_optimal_translation(
            gen_climbs, offset_manifold, x_offsets, y_offsets
        )
        print(f"Best Dx:{best_dx}, Best Dy: {best_dy}")

        # Apply per-climb optimal translation  (B, S, H)
        B, S, H = gen_climbs.shape
        translation = torch.zeros(B, 1, H, device=gen_climbs.device)
        translation[:, 0, 0] = best_dx   # x col
        translation[:, 0, 1] = best_dy   # y col  (pull_x/pull_y cols 2,3 left alone)
        translated_climbs = gen_climbs + translation

        # Now do the standard index projection on the translated climbs
        return self._project_onto_indices(translated_climbs, offset_manifold, layout_id)

    def _project_onto_indices(
            self,
            gen_climbs: Tensor,
            offset_manifold: Tensor,
            layout_id: str,
        ):
        """Project climb onto the final hold indices (and remove null holds)"""
        
        B, S, H = gen_climbs.shape

        flat_climbs = gen_climbs.reshape(-1,H)                  # (B*S, H)

        dists = torch.cdist(flat_climbs[:,:,:(H-NUM_ROLES)]*self.feature_weights, offset_manifold*self.feature_weights)       # (B*S, H, M)
        idx = dists.argmin(dim=1)
        holds = self.holds_lookup[layout_id][idx]
        holds = holds.reshape(B, S)
        
        roles = np.argmax(gen_climbs[:,:,(H-NUM_ROLES):].detach().numpy(),axis=2)
        climbs = np.stack([holds, roles], axis=2)
        print(f"climbs.shape: {climbs.shape}")
        deduped_climbs = []
        for c in climbs:
            valid_mask = c[:, 1] != 4
            c_valid = c[valid_mask]
            c_sorted = c_valid[c_valid[:, 1].argsort()]
            _, unique_indices = np.unique(c_sorted[:, 0], return_index=True)
            deduped_climbs.append(c_sorted[unique_indices].tolist())

        return deduped_climbs
    
    def _projection_strength(self, t: Tensor, t_start_projection: float):
        """Calculate the weight to assign to the projected holds based on the timestep."""
        a = (t_start_projection-t)/t_start_projection
        strength = 1 - torch.cos(a*torch.pi/2)
        return torch.where(t > t_start_projection, torch.zeros_like(t), strength).unsqueeze(2)
    
    @torch.no_grad()
    def generate(
        self,
        layout_id: str,
        n: int,
        angle: int,
        grade: str,
        diff_scale: str,
        timesteps: int,
        deterministic: bool,
        t_start_projection: float,
        x_offset: float | None,
        seed: int,
    )->list[list[list[int]]]:
        """
        Generate a climb or batch of climbs with the given conditions using the standard DDPM iterative denoising process.

        :param layout_id: The Wall-ID on which to generate the climb.
        :type layout_id: str
        :param n: The number of climbs to generate.
        :type n: int
        :param angle: The current wall angle.
        :type angle: int
        :param grade: The desired grade.
        :type grade: str
        :param diff_scale: The desired difficulty scale (V-scale or Font).
        :type diff_scale: str
        :param timesteps: Model setting: Number of diffusion timesteps to run. Higher results in better quality (Should be a divisor of 100 to retain markovian properties)
        :type timesteps: int
        :param deterministic: Whether to use the original noise vector in successive diffusion steps, or use a new noise vector each time.
        :type deterministic: bool
        :param t_start_projection: Point in the generation process to begin the projection steps. Earlier is better but more expensive.
        :type t_start_projection: float
        :param x_offset: Offset the climb on the X-axis.
        :type x_offset: float | None
        :return: A set of generated climbs according to the specified 
        :rtype: list[list[list[int]]]
        """
        # Seed Noise Generator
        if deterministic:
            self.deterministic_noise_generator.manual_seed(seed)
        
        # Handle manifold offset
        auto = True if x_offset is None else False
        offset_manifold = self._get_offset_manifold(layout_id, x_offset)

        # CORE LOGIC
        cond_t = self._build_cond_tensor(n, grade, diff_scale, angle)
        x_t = torch.randn((n, 20, 4), device=self.device, generator=self.deterministic_noise_generator) if deterministic else torch.randn((n, 20, 4), device=self.device)
        noisy = x_t.clone()
        t_tensor = torch.ones((n,1), device=self.device)
        
        for _ in range(0, timesteps):
            gen_climbs = self.ddpm(noisy, cond_t, t_tensor)

            if t_tensor[0].item() < t_start_projection: # This block might be problematic. If not projecting, don't enter it at all.
                alpha_p = self._projection_strength(t_tensor, t_start_projection)
                projected_climbs = self._project_onto_manifold(gen_climbs, offset_manifold)
                gen_climbs = alpha_p*(projected_climbs) + (1-alpha_p)*(gen_climbs)
            
            t_tensor -= 1.0/timesteps
            noisy = self.ddpm.forward_diffusion(gen_climbs, t_tensor, x_t if deterministic else torch.randn_like(x_t))
        
        if auto:
            return self._project_onto_indices_with_translation(gen_climbs, offset_manifold, layout_id)
        return self._project_onto_indices(gen_climbs, offset_manifold, layout_id)

# ---------------------------------------------------------------------------
# Global ClimbGenerator Instance For Dependency Injection
# ---------------------------------------------------------------------------
scaler = ClimbsFeatureScaler(
    weights_path=SCALER_WEIGHTS_PATH
)
ddpm = ClimbDDPM(
    model=Noiser(),
    weights_path=DDPM_WEIGHTS_PATH,
)
hold_classifier = UNetHoldClassifierLogits(
    weights_path=HC_WEIGHTS_PATH
)

# Set to Eval() and compile
ddpm.eval()
# ddpm = torch.compile(ddpm)
hold_classifier.eval()
# hold_classifier = torch.compile(hold_classifier)

generator = ClimbDDPMGenerator(
    scaler=scaler,
    ddpm=ddpm,
    hold_classifier=hold_classifier
)

generator.get_hold_means()

for _ in range(1):
    generator.generate(
        "wall-eadd9a49cd2b", 1, 45, "V4", "v_grade", 100, False, 0.0, None, 37
    )

c:\Users\EvanM\Documents\Projects\GitHub\ml-homewall-climb-generator\model-training\simple_diffusion.py:611: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch

Wall-id--wall-9e000814e880; Means-- x:-0.2532232403755188, y:0.07008305937051773, Px:0.0031096742022782564, Py:-0.3763650953769684 
Wall-id--wall-443c15cd12e0; Means-- x:-0.25757840275764465, y:-0.015613225288689137, Px:0.002315997611731291, Py:-0.32472988963127136 
Wall-id--wall-0a877f13d8e5; Means-- x:-0.2587909400463104, y:-0.023581353947520256, Px:0.003931401297450066, Py:-0.3435281217098236 
Wall-id--wall-eadd9a49cd2b; Means-- x:-0.4820115566253662, y:-0.16430433094501495, Px:-0.024892933666706085, Py:-0.2246631681919098 
Wall-id--wall-e77a688049fb; Means-- x:0.0008089139591902494, y:-0.02644038014113903, Px:-0.003329114057123661, Py:-0.29467177391052246 
X-offset: 0.4819999933242798, Y-offset: 0.164000004529953
New Means-- x:-1.1568910849746317e-05, y:-0.0003043315082322806
tensor([[[1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0.,
          0., 0., 0.]]])
tensor([[0.4964, 0.5136, 0.5215, 0.3677, 0.2852, 0.3386, 0.1197, 0.0788, 0.1732,
         0.4079, 0.2754, 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

def animate_climb_generation(guesses: list, filename="climb_diffusion_no_projection.gif", title="Diffusion Process, V4 @45*, No Projection", fps=15):
    """
    Creates a GIF animating the DDPM denoising process.
    
    Args:
        guesses (list): A list of numpy arrays. Each array should be shape (20, 4) 
                        representing [x, y, pull_x, pull_y] at a specific timestep.
        filename (str): Output filename for the GIF.
        fps (int): Frames per second.
    """
    n_holds = len(guesses[0])
    # Setup the figure and axis
    fig, ax = plt.subplots(figsize=(6, 8))
    
    # We set limits slightly wider than the standard -1,1 to see the noise coalescing
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    ax.set_aspect('equal')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.set_xlabel("X Position (Normalized)")
    ax.set_ylabel("Y Position (Normalized)")
    
    # Initialize graphic elements
    # Scatter plot for holds
    # We initialize with dummy data; 'c' and 's' will be updated in the loop
    dummy = np.zeros(n_holds)
    scat = ax.scatter(dummy, dummy, c=dummy, s=dummy, cmap='viridis', vmin=0, vmax=1, zorder=2)
    
    # Quiver plot for pull vectors
    # angles='xy', scale_units='xy', scale=0.5 matches your plot_climb logic
    quiv = ax.quiver(dummy, dummy, dummy, dummy, 
                     color='green', alpha=0.6,
                     angles='xy', scale_units='xy', scale=0.5,
                     width=0.005, headwidth=3, zorder=1)

    def init():
        """Initialize empty data for the animation."""
        scat.set_offsets(np.empty((0, 2)))
        quiv.set_offsets(np.empty((0, 2)))
        quiv.set_UVC(dummy, dummy)
        return scat, quiv

    def update(frame_idx):
        """Update function for each frame of the animation."""
        # Get data for the current timestep
        # Data shape: (20, 4) -> [x, y, pull_x, pull_y]
        data = guesses[frame_idx]
        
        x = data[:, 0]
        y = data[:, 1]
        pull_x = data[:, 2]
        pull_y = data[:, 3]

        # Determine which holds act as "hands" (significant pull) vs "feet"/smears
        # This matches your plot_climb logic
        is_hand = ((np.absolute(pull_y) > 0.2) | (np.absolute(pull_x) > 0.2))
        
        # Calculate sizes based on type
        sizes = np.full(len(is_hand), 20) + np.full(len(is_hand), 30) * is_hand.astype(float)
        
        # Update Scatter (Positions, Sizes, Colors)
        # Note: set_offsets expects an (N, 2) array
        scat.set_offsets(np.c_[x, y])
        scat.set_sizes(sizes)
        # We use the boolean is_hand array for color mapping (0=Purple, 1=Yellow by default)
        scat.set_array(is_hand.astype(float))
        
        # Update Quiver (Positions and Vectors)
        quiv.set_offsets(np.c_[x, y])
        # Apply the 0.2 scaling factor from your reference code
        quiv.set_UVC(pull_x * 0.2, pull_y * 0.2)
        
        # Update Title
        # Reverse the count so it looks like a countdown (Noise -> Clean) or standard count
        ax.set_title(f"{title}\n t={len(guesses)-frame_idx}/{len(guesses)}")
        
        return scat, quiv

    # Create the animation
    print(f"Generating animation with {len(guesses)} frames...")
    anim = FuncAnimation(fig, update, frames=len(guesses), init_func=init, blit=False)
    
    # Save to GIF
    writer = PillowWriter(fps=fps)
    anim.save(filename, writer=writer)
    print(f"Animation saved to {filename}")

animate_climb_generation(guesses, filename="strong_projection.gif", title="Diffusion Process, V4 @45*, Early/Strong Projection")